# V3 Market-Package Readiness Companion

## tl;dr

The authoritative deterministic builder reports a failed V3 intake: 5/138 acquisition tickers, 4,825/96,500 active member/session keys with bars, no explicit trading-status package, and no authoritative licensed calendar/action/symbol/license binding. This notebook reproduces those calculations without enabling model input or training.

## Context & Methods

The unit of analysis is a point-in-time SET100 date/ticker key. The acquisition union is used only as a provider coverage contract. The V3 membership panel defines which tickers are active on each provisional observed session.

### Key Assumptions

- The checksum-bound membership panel is trusted structural evidence.
- The current Yahoo snapshot is assessed only as a private-pilot gap, not as a licensed package.
- Missing status, calendar, action, symbol, or license evidence fails closed.
- No result in this notebook is a model-performance result.

## Data

### 1. Load the current inputs

In [ ]:
from pathlib import Path
import pandas as pd

from src.v3.market_data_readiness import assess_v3_market_package

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

bars = pd.read_csv(project_root / "data/raw/prices_real_ohlcv.csv")
membership = pd.read_csv(
    project_root
    / "data/processed/v3_development/set100_membership_session_mask_2021_2024.csv.gz"
)
universe = pd.read_csv(
    project_root / "data/reference/set50_or_set100_universe.csv"
)

{
    "bar_rows": len(bars),
    "bar_dates": pd.to_datetime(bars["date"]).nunique(),
    "bar_tickers": bars["ticker"].nunique(),
    "membership_rows": len(membership),
    "active_membership_keys": int(membership["is_index_member"].sum()),
    "acquisition_tickers": universe["ticker"].nunique(),
}

## Results

### 2. Run the fail-closed assessment

In [ ]:
receipt, checks, coverage = assess_v3_market_package(
    bars,
    membership,
    universe,
    generated_at_utc="2026-07-31T00:00:00+00:00",
)

coverage[
    ["metric", "numerator", "denominator", "coverage_percent"]
].round({"coverage_percent": 2})

### 3. Inspect the blocking checks

In [ ]:
checks.loc[
    checks["status"].ne("pass"),
    ["check", "severity", "requirement", "details"],
].reset_index(drop=True)

In [ ]:
{
    "status": receipt["status"],
    "licensed_market_package_eligible": receipt[
        "licensed_market_package_eligible"
    ],
    "model_input_eligible": receipt["model_input_eligible"],
    "training_enabled": receipt["training_enabled"],
    "hpc_jobs_submitted": receipt["hpc_jobs_submitted"],
}

## Takeaways

1. The current five-ticker snapshot is only 3.62% of the acquisition universe and 5.00% of active membership keys.
2. Missing daily status and authoritative calendar evidence prevents missing bars from being interpreted safely.
3. Corporate actions, stable instrument history, and license/checksum bindings remain absent.
4. The V3 intake gate should be rerun with `--candidate-manifest ... --require-pass` after the provider package arrives.
5. A passing intake still does not unlock training until the investability policy and V3 protocol are approved and locked.